In [ ]:
import cv2
import os
import math
import numpy as np

def rotate_360_no_crop_aspect(image_path, output_dir, prefix="train3", border_trim=2):
    os.makedirs(output_dir, exist_ok=True)
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Could not read image")

    h, w = img.shape[:2]
    if border_trim > 0:
        img_trimmed = img[border_trim:h-border_trim, border_trim:w-border_trim]
    else:
        img_trimmed = img

    h_trim, w_trim = img_trimmed.shape[:2]
    corners = np.array([
        [-w_trim/2, -h_trim/2],
        [ w_trim/2, -h_trim/2],
        [ w_trim/2,  h_trim/2],
        [-w_trim/2,  h_trim/2]
    ])
    angles = np.deg2rad(np.arange(0, 360, 1))
    max_w, max_h = 0, 0

    for a in angles:
        R = np.array([[np.cos(a), -np.sin(a)],
                      [np.sin(a),  np.cos(a)]])
        rotated_corners = corners @ R.T
        w_rot = rotated_corners[:,0].max() - rotated_corners[:,0].min()
        h_rot = rotated_corners[:,1].max() - rotated_corners[:,1].min()
        max_w = max(max_w, w_rot)
        max_h = max(max_h, h_rot)

    new_w = int(np.ceil(max_w))
    new_h = int(np.ceil(max_h))

    center = (new_w // 2, new_h // 2)
    bg_color = (83, 0, 68)
    canvas = np.full((new_h, new_w, 3), bg_color, dtype=np.uint8)
    y = (new_h - h_trim) // 2
    x = (new_w - w_trim) // 2
    canvas[y:y+h_trim, x:x+w_trim] = img_trimmed
    for angle in range(360):
        M = cv2.getRotationMatrix2D(center, angle, 1.0)

        rotated = cv2.warpAffine(
            canvas,
            M,
            (new_w, new_h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=bg_color
        )

        cv2.imwrite(
            os.path.join(output_dir, f"{prefix}_{angle:03d}.png"),
            rotated
        )

    print(f"360 rotation completed with border trimmed={border_trim}, custom background, and original aspect ratio preserved")


In [84]:
rotate_360_no_crop_aspect(
    image_path="quantom_dataset/train/train3.png",
    output_dir="augmented_images"
)

✅ 360° rotation completed with border trimmed=2, custom background, and original aspect ratio preserved


In [80]:
import cv2
import numpy as np
import os

def translate_image(image_path, output_dir, prefix="translate_train1", max_shift=10, border_trim=2):
    os.makedirs(output_dir, exist_ok=True)
    
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Could not read image")
    
    h, w = img.shape[:2]
    if border_trim > 0:
        img_trimmed = img[border_trim:h-border_trim, border_trim:w-border_trim]
    else:
        img_trimmed = img

    h_trim, w_trim = img_trimmed.shape[:2]
    bg_color = (83, 0, 68)
    shifts = [
        ( max_shift, 0),
        (-max_shift, 0),
        (0,  max_shift),
        (0, -max_shift),
        ( max_shift,  max_shift),
        (-max_shift,  max_shift),
        ( max_shift, -max_shift),
        (-max_shift, -max_shift),
        (0, 0)
    ]
    
    for i, (dx, dy) in enumerate(shifts):
        canvas = np.full((h_trim, w_trim, 3), bg_color, dtype=np.uint8)
        x_start = max(0, dx)
        y_start = max(0, dy)
        x_end = min(w_trim, w_trim + dx) if dx < 0 else min(w_trim, w_trim - dx + w_trim)
        y_end = min(h_trim, h_trim + dy) if dy < 0 else min(h_trim, h_trim - dy + h_trim)
        src_x_start = max(0, -dx)
        src_y_start = max(0, -dy)
        src_x_end = src_x_start + (x_end - x_start)
        src_y_end = src_y_start + (y_end - y_start)
        canvas[y_start:y_end, x_start:x_end] = img_trimmed[src_y_start:src_y_end, src_x_start:src_x_end]
        out_path = os.path.join(output_dir, f"{prefix}_{i+9:02d}.png")
        cv2.imwrite(out_path, canvas)
    
    print(f"Translated images saved to {output_dir} with border_trim={border_trim}")


In [ ]:
test_path = 'train_images/translate_train(2)_0_11.png'

360

In [112]:
train3_images = [os.path.join('augmented_images/train3' , img ) for img in os.listdir('augmented_images/train3')]

for i , img in  enumerate(train3_images) : 
    random_max_shift = [35, 65 ,95]
    random_index = np.random.randint(0 , 3)
    print(img)
    translate_image(img, "output_trans", prefix=f"translate_train(3)_{i}", max_shift=random_max_shift[random_index])


augmented_images/train3/train3_281.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_001.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_110.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_014.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_067.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_192.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_257.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_130.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_016.png
✅ Translated images saved to output_trans with border_trim=2
augmented_images/train3/train3_117.png
✅ Translated images saved to output_trans with border_trim=2


In [68]:
len(os.listdir('augmented_images/train2'))

387

In [113]:
len(os.listdir('train_images'))

9720